In [150]:
%load_ext cudf.pandas

import numpy as np
import pandas as pd


The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


In [151]:
df = pd.read_csv("/kaggle/input/datasets/organizations/crowdflower/twitter-airline-sentiment/Tweets.csv")

In [152]:
df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,None,nan,Virgin America,None,cairdin,None,0,@VirginAmerica What @dhepburn said.,None,2015-02-24 11:35:52 -0800,None,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,None,0.0,Virgin America,None,jnardino,None,0,@VirginAmerica plus you've added commercials t...,None,2015-02-24 11:15:59 -0800,None,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,None,nan,Virgin America,None,yvonnalynn,None,0,@VirginAmerica I didn't today... Must mean I n...,None,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,None,jnardino,None,0,@VirginAmerica it's really aggressive to blast...,None,2015-02-24 11:15:36 -0800,None,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0,Virgin America,None,jnardino,None,0,@VirginAmerica and it's a really big bad thing...,None,2015-02-24 11:14:45 -0800,None,Pacific Time (US & Canada)


## Phase 1 : EDA

### Checking for null values

In [153]:
print(f"TOTAL NUMS OF SAMPLES : {df.shape}\n\n")
print(f"TOTAL NUMS OF NULL VALUES : \n\n{df.isna().sum()}\n\n")
print(f"TOTAL NUMS OF NULL VALUES : \n\n{df.isna().mean()*100}\n")

TOTAL NUMS OF SAMPLES : (14640, 15)


TOTAL NUMS OF NULL VALUES : 

tweet_id                            0
airline_sentiment                   0
airline_sentiment_confidence        0
negativereason                   5462
negativereason_confidence        4118
airline                             0
airline_sentiment_gold          14600
name                                0
negativereason_gold             14608
retweet_count                       0
text                                0
tweet_coord                     13621
tweet_created                       0
tweet_location                   4733
user_timezone                    4820
dtype: int64


TOTAL NUMS OF NULL VALUES : 

tweet_id                         0.000000
airline_sentiment                0.000000
airline_sentiment_confidence     0.000000
negativereason                  37.308743
negativereason_confidence       28.128415
airline                          0.000000
airline_sentiment_gold          99.726776
name                   

Since feature "airline_sentiment_gold", "negativereason_gold", and "tweet_coord" are having more than 90 % of missing values, so we will be removing them.

In [154]:
df_1 = df.drop(columns=['tweet_coord','negativereason_gold','airline_sentiment_gold'])

In [155]:
# checking for remaining null values in df_1 
print(f"TOTAL FEATURES : {df_1.shape[1]}\n\n")
print(f"TOTAL NUMS OF NULL VALUES : \n\n{df_1.isna().sum()}\n\n")
print(f"TOTAL NUMS OF NULL VALUES : \n\n{df_1.isna().mean()*100}\n")

TOTAL FEATURES : 12


TOTAL NUMS OF NULL VALUES : 

tweet_id                           0
airline_sentiment                  0
airline_sentiment_confidence       0
negativereason                  5462
negativereason_confidence       4118
airline                            0
name                               0
retweet_count                      0
text                               0
tweet_created                      0
tweet_location                  4733
user_timezone                   4820
dtype: int64


TOTAL NUMS OF NULL VALUES : 

tweet_id                         0.000000
airline_sentiment                0.000000
airline_sentiment_confidence     0.000000
negativereason                  37.308743
negativereason_confidence       28.128415
airline                          0.000000
name                             0.000000
retweet_count                    0.000000
text                             0.000000
tweet_created                    0.000000
tweet_location                  32.3292

In [156]:
df_1.sample(3)

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,name,retweet_count,text,tweet_created,tweet_location,user_timezone
4016,567872182136619008,negative,1.0000,Can't Tell,0.6584,United,sarahkorich,0,@united how else would I know it was denied?,2015-02-17 18:24:13 -0800,SLC | LA,Pacific Time (US & Canada)
8548,568173792406740994,negative,1.0000,Can't Tell,1.0000,Delta,mcontrerasnyc,0,@JetBlue PR-friendly tweets don't help drive a...,2015-02-18 14:22:42 -0800,None,None
13039,569953628687101954,positive,0.6826,None,0.0000,American,megan_shearin,0,@AmericanAir thank you. They are processing my...,2015-02-23 12:15:08 -0800,Hampton Roads,None


In [157]:
df_1.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   tweet_id                      14640 non-null  int64
 1   airline_sentiment             14640 non-null  object
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object
 6   name                          14640 non-null  object
 7   retweet_count                 14640 non-null  int64
 8   text                          14640 non-null  object
 9   tweet_created                 14640 non-null  object
 10  tweet_location                9907 non-null   object
 11  user_timezone                 9820 non-null   object
dtypes: float64(2), int64(2), object(8)
memory usage: 3.5+ MB


In [158]:
df_1.columns.tolist()

['tweet_id',
 'airline_sentiment',
 'airline_sentiment_confidence',
 'negativereason',
 'negativereason_confidence',
 'airline',
 'name',
 'retweet_count',
 'text',
 'tweet_created',
 'tweet_location',
 'user_timezone']

### Changing the column names

In [159]:
dict = {
    'tweet_id':'id',
     'airline_sentiment':'sentiment',
     'airline_sentiment_confidence':'sentiment_confidence',
     'negativereason':'reason',
     'negativereason_confidence':'reason_confidence',
     'airline':'airlines',
     'name':'name',
     'retweet_count':'retweets',
     'text':'feedback',
     'tweet_created':'date_time',
     'tweet_location':'location',
     'user_timezone':'timezone'
    }


df_1 = df_1.rename(columns=dict)

In [160]:
df_1.columns.tolist()

['id',
 'sentiment',
 'sentiment_confidence',
 'reason',
 'reason_confidence',
 'airlines',
 'name',
 'retweets',
 'feedback',
 'date_time',
 'location',
 'timezone']

In [161]:
df_1 = df_1[['id','name','date_time', 'location', 'timezone','airlines','feedback','reason','reason_confidence','retweets','sentiment','sentiment_confidence']]
df_1.sample(5)

,id,name,date_time,location,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment,sentiment_confidence
776,570085448271335424,tomcrabtree,2015-02-23 20:58:57 -0800,San Francisco,Pacific Time (US & Canada),United,@united Lost bags. Cancelled Flightled flights...,Cancelled Flight,0.6567,1,negative,1.0000
9972,569596783011217408,CaseyRhoades1,2015-02-22 12:37:10 -0800,"Madbury, NH",None,US Airways,@USAirways 2nd plane forced to get off due to ...,Cancelled Flight,0.3367,0,negative,0.6633
2766,568913508936458240,AtulKC,2015-02-20 15:22:05 -0800,Greater New York City Area,Eastern Time (US & Canada),United,@united - terrible experience on UA415 on 17th...,Bad Flight,0.6907,0,negative,1.0000
2566,569058014650433536,_SamanthaAkira,2015-02-21 00:56:17 -0800,None,Pacific Time (US & Canada),United,@united no. U guys suck. I'll never fly with u...,Flight Attendant Complaints,1.0000,0,negative,1.0000
14299,569640038302134273,manuel_c,2015-02-22 15:29:03 -0800,USA,Eastern Time (US & Canada),American,"As am I, @AmericanAir - but thankfully there w...",Can't Tell,0.6676,0,negative,1.0000


In [162]:
df_1.isna().sum()

id                         0
name                       0
date_time                  0
location                4733
timezone                4820
airlines                   0
feedback                   0
reason                  5462
reason_confidence       4118
retweets                   0
sentiment                  0
sentiment_confidence       0
dtype: int64

In [163]:
df_1.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id                    14640 non-null  int64
 1   name                  14640 non-null  object
 2   date_time             14640 non-null  object
 3   location              9907 non-null   object
 4   timezone              9820 non-null   object
 5   airlines              14640 non-null  object
 6   feedback              14640 non-null  object
 7   reason                9178 non-null   object
 8   reason_confidence     10522 non-null  float64
 9   retweets              14640 non-null  int64
 10  sentiment             14640 non-null  object
 11  sentiment_confidence  14640 non-null  float64
dtypes: float64(2), int64(2), object(8)
memory usage: 3.5+ MB


In [164]:
df_1.dtypes.unique()

array([dtype('int64'), dtype('O'), dtype('float64')], dtype=object)

In [165]:
df_1.dtypes[df_1.dtypes == 'object'].reset_index()

,index,0
0,name,object
1,date_time,object
2,location,object
3,timezone,object
4,airlines,object
5,feedback,object
6,reason,object
7,sentiment,object


In [166]:
df_1.dtypes[df_1.dtypes == 'int'].reset_index()

,index,0
0,id,int64
1,retweets,int64


In [167]:
df_1.dtypes[df_1.dtypes == 'float'].reset_index()

,index,0
0,reason_confidence,float64
1,sentiment_confidence,float64


In [168]:
pd.DataFrame({'Missing Values': df_1.isna().sum(), 'Data Type': df_1.dtypes})

,Missing Values,Data Type
id,0,int64
name,0,object
date_time,0,object
location,4733,object
timezone,4820,object
airlines,0,object
feedback,0,object
reason,5462,object
reason_confidence,4118,float64
retweets,0,int64


In [169]:
df_1.nunique()

id                      14485
name                     7701
date_time               14247
location                 3081
timezone                   85
airlines                    6
feedback                14427
reason                     10
reason_confidence        1410
retweets                   18
sentiment                   3
sentiment_confidence     1023
dtype: int64

In [170]:
import datetime as dt

In [171]:
# converting date_time into datetime format
df_1['date_time'] = pd.to_datetime(df_1['date_time'])
df_1.dtypes

id                                          int64
name                                       object
date_time               datetime64[ns, UTC-08:00]
location                                   object
timezone                                   object
airlines                                   object
feedback                                   object
reason                                     object
reason_confidence                         float64
retweets                                    int64
sentiment                                  object
sentiment_confidence                      float64
dtype: object

In [172]:
df_1['date'] = df_1['date_time'].dt.date
df_1['time'] = df_1['date_time'].dt.time

In [173]:
df_1 = df_1[['id','name','date','time', 'timezone','airlines','feedback','reason','reason_confidence','retweets','sentiment','sentiment_confidence']]
df_1.sample(5)

,id,name,date,time,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment,sentiment_confidence
9459,569926031542345728,marvinatorsb,2015-02-23,10:25:29,None,US Airways,"@USAirways booked flight 1mnth ago, seat CONFI...",Customer Service Issue,0.6598,0,negative,1.0000
3535,568283819477839872,Hanagavadi,2015-02-18,21:39:55,None,United,@united ... horrible wait time on 4348 to get ...,Late Flight,0.6816,0,negative,1.0000
12229,570257935911096320,towbinator,2015-02-24,08:24:21,Eastern Time (US & Canada),American,@AmericanAir I'm sure they did. It's certainly...,None,NaN,0,neutral,0.6725
9596,569849281533874176,LaurenCarrot,2015-02-23,05:20:30,None,US Airways,@USAirways I'm referring to email like this. ...,None,0.0000,0,neutral,0.7006
3879,568057226776219648,sasharashid,2015-02-18,06:39:31,Eastern Time (US & Canada),United,@united 1627 to Montego Bay going back to gate...,Customer Service Issue,0.6772,0,negative,1.0000


### Fixing the TimeZones

In [174]:
df_1['timezone'].unique().tolist()

['Eastern Time (US & Canada)',
 'Pacific Time (US & Canada)',
 'Central Time (US & Canada)',
 'America/New_York',
 'Atlantic Time (Canada)',
 'Quito',
 None,
 'Mountain Time (US & Canada)',
 'Vienna',
 'Caracas',
 'Kuala Lumpur',
 'Brisbane',
 'Arizona',
 'London',
 'Tehran',
 'Alaska',
 'Sydney',
 'Irkutsk',
 'Santiago',
 'Amsterdam',
 'Tijuana',
 'Abu Dhabi',
 'Central America',
 'Edinburgh',
 'Jerusalem',
 'Hawaii',
 'Paris',
 'Guam',
 'New Delhi',
 'Stockholm',
 'America/Chicago',
 'Berlin',
 'Madrid',
 'Athens',
 'Brussels',
 'Taipei',
 'Rome',
 'Beijing',
 'Mexico City',
 'Bern',
 'Singapore',
 'Indiana (East)',
 'Melbourne',
 'Saskatchewan',
 'Casablanca',
 'Brasilia',
 'Kyiv',
 'Bucharest',
 'Greenland',
 'Prague',
 'New Caledonia',
 'Bogota',
 'Seoul',
 'Sarajevo',
 'Wellington',
 'Bangkok',
 'Warsaw',
 'Copenhagen',
 'Hong Kong',
 'Guadalajara',
 'Mid-Atlantic',
 'Mazatlan',
 'Buenos Aires',
 'America/Los_Angeles',
 'Dublin',
 'Lisbon',
 'Newfoundland',
 'Monterrey',
 'Tokyo'

In [175]:
df_1['timezone'] = df_1['timezone'].str.replace(" (US & Canada)", "")
df_1['timezone'] = df_1['timezone'].str.replace(" ", "_")

In [176]:
# other way to do the same as above
timezone_fixes = {
    # Fixing the Eastern Time duplicates
    'America/New_York': 'Eastern_Time',
    'America/Detroit': 'Eastern_Time',
    'EST': 'Eastern_Time',
    'Indiana_(East)': 'Eastern_Time',
    
    # Fixing the Central Time duplicates
    'America/Chicago': 'Central_Time',
    
    # Fixing the Mountain Time duplicates
    'America/Boise': 'Mountain_Time',
    
    # Fixing the Pacific Time duplicates
    'America/Los_Angeles': 'Pacific_Time'
}

df_1['timezone'] = df_1['timezone'].replace(timezone_fixes)

In [177]:
df_1['timezone'].unique()

array(['Eastern_Time', 'Pacific_Time', 'Central_Time',
       'Atlantic_Time_(Canada)', 'Quito', None, 'Mountain_Time', 'Vienna',
       'Caracas', 'Kuala_Lumpur', 'Brisbane', 'Arizona', 'London',
       'Tehran', 'Alaska', 'Sydney', 'Irkutsk', 'Santiago', 'Amsterdam',
       'Tijuana', 'Abu_Dhabi', 'Central_America', 'Edinburgh',
       'Jerusalem', 'Hawaii', 'Paris', 'Guam', 'New_Delhi', 'Stockholm',
       'Berlin', 'Madrid', 'Athens', 'Brussels', 'Taipei', 'Rome',
       'Beijing', 'Mexico_City', 'Bern', 'Singapore', 'Melbourne',
       'Saskatchewan', 'Casablanca', 'Brasilia', 'Kyiv', 'Bucharest',
       'Greenland', 'Prague', 'New_Caledonia', 'Bogota', 'Seoul',
       'Sarajevo', 'Wellington', 'Bangkok', 'Warsaw', 'Copenhagen',
       'Hong_Kong', 'Guadalajara', 'Mid-Atlantic', 'Mazatlan',
       'Buenos_Aires', 'Dublin', 'Lisbon', 'Newfoundland', 'Monterrey',
       'Tokyo', 'Midway_Island', 'Istanbul', 'Solomon_Is.',
       'America/Atikokan', 'Adelaide', 'Nairobi', 'Lima', 'Is

### Fixing the Airlines

In [178]:
df_1['airlines'].unique()

array(['Virgin America', 'United', 'Southwest', 'Delta', 'US Airways',
       'American'], dtype=object)

In [179]:
df_1['airlines'] = df_1['airlines'].str.replace(" ", "_")

In [180]:
df_1['airlines'].unique()

array(['Virgin_America', 'United', 'Southwest', 'Delta', 'US_Airways',
       'American'], dtype=object)

In [184]:
pd.DataFrame({'MISSING VALUES':df_1.isna().sum(), 'DTYPES':df_1.dtypes})

,MISSING VALUES,DTYPES
id,0,int64
name,0,object
date,0,object
time,0,object
timezone,4820,object
airlines,0,object
feedback,0,object
reason,5462,object
reason_confidence,4118,float64
retweets,0,int64


In [188]:
df_1['date'] = pd.to_datetime(df_1['date'])
df_1['hour'] = df_1['date'].dt.hour


In [190]:
df_1 = df_1[['id','name','date','hour', 'timezone','airlines','feedback','reason','reason_confidence','retweets','sentiment','sentiment_confidence']]


In [192]:
pd.DataFrame({'MISSING VALUES':df_1.isna().sum(), 'UNIQUE VALUES':df_1.nunique(), 'DTYPES':df_1.dtypes})

,MISSING VALUES,UNIQUE VALUES,DTYPES
id,0,14485,int64
name,0,7701,object
date,0,9,datetime64[ns]
hour,0,1,int32
timezone,4820,78,object
airlines,0,6,object
feedback,0,14427,object
reason,5462,10,object
reason_confidence,4118,1410,float64
retweets,0,18,int64


In [193]:
df_1

,id,name,date,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment,sentiment_confidence
0,570306133677760513,cairdin,2015-02-24,0,Eastern_Time,Virgin_America,@VirginAmerica What @dhepburn said.,None,NaN,0,neutral,1.0000
1,570301130888122368,jnardino,2015-02-24,0,Pacific_Time,Virgin_America,@VirginAmerica plus you've added commercials t...,None,0.0000,0,positive,0.3486
2,570301083672813571,yvonnalynn,2015-02-24,0,Central_Time,Virgin_America,@VirginAmerica I didn't today... Must mean I n...,None,NaN,0,neutral,0.6837
3,570301031407624196,jnardino,2015-02-24,0,Pacific_Time,Virgin_America,@VirginAmerica it's really aggressive to blast...,Bad Flight,0.7033,0,negative,1.0000
4,570300817074462722,jnardino,2015-02-24,0,Pacific_Time,Virgin_America,@VirginAmerica and it's a really big bad thing...,Can't Tell,1.0000,0,negative,1.0000
...,...,...,...,...,...,...,...,...,...,...,...,...
14635,569587686496825344,KristenReenders,2015-02-22,0,None,American,@AmericanAir thank you we got on a different f...,None,0.0000,0,positive,0.3487
14636,569587371693355008,itsropes,2015-02-22,0,None,American,@AmericanAir leaving over 20 minutes Late Flig...,Customer Service Issue,1.0000,0,negative,1.0000
14637,569587242672398336,sanyabun,2015-02-22,0,None,American,@AmericanAir Please bring American Airlines to...,None,NaN,0,neutral,1.0000
14638,569587188687634433,SraJackson,2015-02-22,0,Eastern_Time,American,"@AmericanAir you have my money, you change my ...",Customer Service Issue,0.6659,0,negative,1.0000
